# Phase 2 - B4 GCR-style Gradient Coreset Replay (Kaggle)

Runs the selected frozen XNLI orders with the shared Phase 1 candidate-label prompt and LoRA trainer. After each language stage, B4 computes supervised answer-token cross-entropy gradients for all current-stage examples, projects them into a fixed seeded 256-dimensional signed CountSketch, and greedily herds the prior memory plus current candidates into a 125-example coreset. Later stages sample one replay example with replacement per current example. This is a documented GCR-style adaptation, not an exact reproduction of the GCR paper.

**Compute:** each order adds 2,500 per-example gradient backward passes and trains on 4,500 examples (B0 trains on 2,500). The prior T4 B0 runs took about 46 minutes/order; B4 has not yet been benchmarked. Running all five B4 orders may approach Kaggle's session limit. For safer runs, choose one or two order IDs, run them, save a Kaggle notebook version/output, attach that output to the next session, and set `PREVIOUS_ARTIFACTS_DIR` below to its `phase2_artifacts` folder. Completed orders are validated and skipped; an interrupted order restarts from that order's beginning.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
3. This notebook clones the public project; no GitHub token is needed.
4. For a first run, leave `PREVIOUS_ARTIFACTS_DIR = None`. For later sessions, attach the prior Kaggle output as notebook input and set the variable to its `phase2_artifacts` folder.
5. Run all cells, then use **Save Version** with notebook output enabled to preserve `/kaggle/working/phase2_artifacts`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if REPO_ROOT.exists():
    raise FileExistsError(f"Repository directory already exists: {REPO_ROOT}. Restart the Kaggle session or choose a fresh path.")
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)], check=True)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B4 runner: {runner_path}")
help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
if help_result.returncode != 0 or "B4" not in help_result.stdout:
    raise RuntimeError("The cloned GitHub revision does not expose B4. Check that the B4 runner commit is on the default branch.")

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_artifacts")
PREVIOUS_ARTIFACTS_DIR = None  # Example: Path("/kaggle/input/previous-b4-output/phase2_artifacts")
print({"repo": str(REPO_ROOT), "revision": repo_revision, "artifact_root": str(ARTIFACT_ROOT), "runner_file": str(runner_path)})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))
protocol = load_protocol(PROTOCOL_PATH)

if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if ARTIFACT_ROOT.exists():
        raise FileExistsError("A partial working artifact directory exists without a protocol manifest. Inspect it before continuing.")
    if PREVIOUS_ARTIFACTS_DIR is not None:
        previous = Path(PREVIOUS_ARTIFACTS_DIR)
        if not (previous / "protocol_manifest.json").is_file():
            raise FileNotFoundError(f"Previous artifact manifest not found under {previous}")
        shutil.copytree(previous, ARTIFACT_ROOT)
        print("Copied previous Kaggle artifacts from:", previous)
    else:
        prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
        print("Prepared the fixed XNLI subsets.")
prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)

ORDER_IDS = ["order-1"]  # Change to selected IDs, e.g. ["order-1", "order-2"], or all five.
valid_order_ids = {order["id"] for order in protocol["orders"]}
if not ORDER_IDS or len(set(ORDER_IDS)) != len(ORDER_IDS) or set(ORDER_IDS) - valid_order_ids:
    raise ValueError(f"Choose unique protocol order IDs from {sorted(valid_order_ids)}.")
print({"languages": protocol["dataset"]["languages"], "orders_to_run": ORDER_IDS, "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "gradient_passes_per_order": protocol["dataset"]["train_examples_per_language"] * len(protocol["dataset"]["languages"]), "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})


In [ ]:
command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method", "B4",
    "--protocol", str(PROTOCOL_PATH),
    "--artifact-root", str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import statistics
import json

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {run["run_id"]: run for run in run_manifest["runs"] if run["method_id"] == "B4"}
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if result_path.is_file():
        record = json.loads(result_path.read_text(encoding="utf-8"))
        validate_result_record(record, run_manifest)
        completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B4",
    "method_interpretation": "GCR-style gradient-herding adaptation; not an exact reproduction",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "complete": len(completed) == len(expected_runs),
    "per_order": {order_id: {name: record["metrics"][name] for name in metric_names} for order_id, record in sorted(completed.items())},
}
if completed:
    summary["across_order_summary"] = {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    }
summary_path = ARTIFACT_ROOT / "B4_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"Validated {len(completed)}/{len(expected_runs)} B4 orders. Save the Kaggle notebook version/output to preserve artifacts:", ARTIFACT_ROOT)


## Saved outputs and continuing in another Kaggle session

- `/kaggle/working/phase2_artifacts/protocol_manifest.json` and `data/`: fixed selected examples and checksums.
- `phase2_artifacts/checkpoints/<run_id>/`: adapters/tokenizers after each language stage.
- `phase2_artifacts/ledgers/<run_id>/`: evaluation matrix and ledger.
- `phase2_artifacts/results/<run_id>.json`: validated result for each completed order, including gradient counts/timing and replay composition.
- `phase2_artifacts/B4_summary.json`: validated results and summary; `complete` is true only after all five orders.

To continue in a new Kaggle session, attach the previous notebook output as an input, set `PREVIOUS_ARTIFACTS_DIR` to the mounted `phase2_artifacts` directory, choose the next `ORDER_IDS`, then save the new output again. Keep the same data manifest and code revision for every order. Never tune using test scores.
